# Descriptives of the analysis samples

Table 1 and Appendix Tables B1 and B2: the onboarding sample (N = 74) and the offboarding
sample (N = 56), overall and by condition.

In [ ]:
import re
from pathlib import Path

import pandas as pd
import pingouin as pg
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt

DATA_DIR = Path('../data')
TABLE_DIR = Path('../tables_figures')

## Variables

In [ ]:
# Education in three levels: sd_edu_be_rec code -> level.
EDU_LEVELS = {1: 1, 2: 1, 3: 1, 4: 2, 5: 2, 6: 3, 7: 3}

CONDITIONS = {'K': 'Control', 'M': 'Experimental'}

# Table rows, in this order. A plain label gives M (SD); a (label, categories) pair gives
# % per category.
PAPER_ROWS = {
    'sd_age_years': 'Age (years)',
    'sd_gender': ('Gender', {1: 'Man', 2: 'Woman', 3: 'Other'}),
    'edu_level': ('Education', {1: 'Secondary not completed', 2: 'Secondary completed',
                                3: 'Higher education'}),
    'news_interest_': 'News interest (1-5)',
    'interest_politics_': 'Political interest (1-5)',
    'ideo_lr_1': 'Left-right ideology (-5 to 5)',
    # The scale means used in the models: diversity-seeking values and behavior over items 1-7.
    'div_values_mean': 'Diversity-seeking values (1-5)',
    'div_traits_mean': 'Diversity-seeking behavior (1-5)',
    'div_perceived_mean': 'Perceived diversity of national news media (1-5)',
    'inst_trust_1': 'Trust in social media companies (1-5)',
    'inst_trust_2': 'Trust in news media (1-5)',
    'internet_use': 'Internet use (1-8)',
    'news_use_6': 'News use via social media (1-7)',
}

## Load data

In [ ]:
def read_sample(path):
    """One row per participant, with education levels added.

    The data have one row per participant-day; survey answers repeat on every row.
    """
    df = pd.read_csv(path, sep=';', decimal=',', encoding='utf-8-sig').drop_duplicates('handle')
    df['edu_level'] = df['sd_edu_be_rec'].map(EDU_LEVELS)
    return df


samples = {
    'Onboarding and active': read_sample(DATA_DIR / 'onboarding_sample_round1.csv'),
    'Also offboarding': read_sample(DATA_DIR / 'offboarding_sample_round1.csv'),
}
for name, df in samples.items():
    print(name, df.shape)

## Tables

In [ ]:
INDENT = '   '  # one level of indentation: variables under a section, categories under a variable


def describe(df, rows):
    """Rows of one table column: M (SD) for a plain label, % per category for a (label, categories) pair."""
    # Lists rather than a dict: category labels such as 'Other' could repeat across variables.
    labels, cells = [], []
    for var, row in rows.items():
        if isinstance(row, tuple):
            label, categories = row
            counts = df[var].value_counts()
            assert set(counts.index) <= set(categories), f'{var}: code without a label'
            labels.append(INDENT + label)
            cells.append('')
            for code, category in categories.items():
                labels.append(2 * INDENT + category)
                cells.append(f'{100 * counts.get(code, 0) / counts.sum():.1f}')
            continue
        values = df[var].dropna()
        labels.append(INDENT + row)
        cells.append(f'{values.mean():.2f} ({values.std():.2f})')
    return pd.Series(cells, index=labels)


def sections(rows):
    """Split rows into titled sections: continuous variables, then categorical variables."""
    categorical = [var for var in rows if isinstance(rows[var], tuple)]
    continuous = [var for var in rows if var not in categorical]
    groups = {'Continuous variables, M (SD)': continuous,
              'Categorical variables, %': categorical}
    return {title: {var: rows[var] for var in group} for title, group in groups.items()}


def make_table(rows, groups, size='N'):
    """One column per group: {column name: dataframe}.

    size is the symbol for the group size in the column head: APA uses N for a whole
    sample and n for a subgroup of it, such as a condition.
    """
    columns = {}
    for name, df in groups.items():
        parts = []
        for title, section in sections(rows).items():
            parts += [pd.Series([''], index=[title]), describe(df, section)]
        columns[f'{name} ({size} = {len(df)})'] = pd.concat(parts)
    first = next(iter(columns.values()))
    return pd.DataFrame({name: column.values for name, column in columns.items()}, index=first.index)

In [ ]:
STATISTICS = re.compile(r'\b(M|SD|N|n)\b')  # APA sets statistical symbols in italics


def apa_text(text):
    """APA typography: an en dash in ranges (1–5) and a minus sign for negatives (−0.76)."""
    text = re.sub(r'(\d)-(\d)', '\\1\u2013\\2', text)
    return re.sub(r'(?<!\w)-(?=\d)', '\u2212', text)


def write(paragraph, text, bold=False, italic=False):
    """Add text to a paragraph, with the statistical symbols in italics."""
    for i, part in enumerate(STATISTICS.split(apa_text(text))):
        if part:
            run = paragraph.add_run(part)
            run.bold = bold
            run.italic = italic or i % 2 == 1  # split() puts the matched symbols at odd positions
    return paragraph


def set_border(cell, side):
    """Draw a single horizontal rule on one side of a cell."""
    properties = cell._tc.get_or_add_tcPr()
    borders = properties.first_child_found_in('w:tcBorders')
    if borders is None:
        borders = OxmlElement('w:tcBorders')
        properties.append(borders)
    line = OxmlElement(f'w:{side}')
    for key, value in {'val': 'single', 'sz': '4', 'space': '0', 'color': '000000'}.items():
        line.set(qn(f'w:{key}'), value)
    borders.append(line)


def save_apa_table(table, path, number, title, note):
    """Save a table as .docx in APA 7 style: bold number, italic title, horizontal rules only, note."""
    doc = Document()
    style = doc.styles['Normal']
    style.font.name = 'Times New Roman'
    style.font.size = Pt(12)
    style.paragraph_format.space_after = Pt(0)

    write(doc.add_paragraph(), f'Table {number}', bold=True)
    write(doc.add_paragraph(), title, italic=True)

    grid = doc.add_table(rows=1, cols=len(table.columns) + 1)  # no table style: no vertical lines
    grid.alignment = WD_TABLE_ALIGNMENT.CENTER
    grid.autofit = False
    # Group sizes on their own header line, e.g. 'Also offboarding' / '(N = 56)' or 'Control' / '(n = 38)'.
    headers = ['Variable', *(re.sub(r' \((?=[Nn] =)', '\n(', column) for column in table.columns)]
    for cell, text in zip(grid.rows[0].cells, headers):
        write(cell.paragraphs[0], text)
        if text != 'Variable':  # APA: the stub head is left-aligned, column heads centred
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
        set_border(cell, 'top')
        set_border(cell, 'bottom')
    for label, values in table.iterrows():
        cells = grid.add_row().cells
        label_paragraph = write(cells[0].paragraphs[0], label.strip())
        # Section titles sit flush left; each INDENT in the label adds one indent level.
        level = (len(label) - len(label.lstrip(' '))) // len(INDENT)
        label_paragraph.paragraph_format.left_indent = Pt(12 * level)
        for cell, value in zip(cells[1:], values):
            write(cell.paragraphs[0], value)
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
    for cell in grid.rows[-1].cells:
        set_border(cell, 'bottom')
    # A wide label column and narrower value columns. Word reads the width of each cell,
    # other editors the table grid, so both are set.
    for i, column in enumerate(grid.columns):
        width = Inches(3.0) if i == 0 else Inches(1.75)
        column.width = width
        for cell in column.cells:
            cell.width = width

    note_paragraph = doc.add_paragraph()
    note_paragraph.paragraph_format.space_before = Pt(6)
    note_paragraph.add_run('Note. ').italic = True
    write(note_paragraph, note)
    doc.save(path)

## Table 1

In [ ]:
TABLE_1_TITLE = 'Sample Characteristics and Descriptive Statistics of the Study Variables'
TABLE_1_NOTE = ('Continuous variables are reported as M (SD) and categorical variables as percentages. '
                'One missing value on age was mean-imputed.')

table_1 = make_table(PAPER_ROWS, samples)
save_apa_table(table_1, TABLE_DIR / 'table_1.docx', 1, TABLE_1_TITLE, TABLE_1_NOTE)
table_1

## Tables B1 and B2: by condition

In [ ]:
APPENDIX_TITLE = 'Sample Characteristics and Descriptive Statistics of the Study Variables by Condition, {} Sample'
# sample -> (table number, survey, who is in the sample)
APPENDIX_TABLES = {
    'Onboarding and active': ('B1', 'onboarding',
                              'participants who completed the onboarding survey and were active on the feed'),
    'Also offboarding': ('B2', 'offboarding', 'participants who also completed the offboarding survey'),
}

for name, (number, survey, sample_description) in APPENDIX_TABLES.items():
    df = samples[name]
    by_condition = {label: df[df['arm'] == code] for code, label in CONDITIONS.items()}
    condition_table = make_table(PAPER_ROWS, by_condition, size='n')
    note = ('Continuous variables are reported as M (SD) and categorical variables as percentages. '
            f'The sample comprises {sample_description} (N = {len(df)}). '
            'One missing value on age, in the experimental condition, was mean-imputed.')
    save_apa_table(condition_table, TABLE_DIR / f'table_{number}.docx', number,
                   APPENDIX_TITLE.format(survey.title()), note)
    display(condition_table)

## Reliability of the diversity-seeking scales

In [ ]:
# Cronbach's alpha of the seven items of each scale, in the onboarding sample.
SCALE_ITEMS = {
    'Values index': [f'div_values_{i}' for i in range(1, 8)],
    'Traits index': [f'div_traits_{i}' for i in range(1, 8)],
}
onboarding = samples['Onboarding and active']
for scale, items in SCALE_ITEMS.items():
    alpha, ci = pg.cronbach_alpha(onboarding[items])
    print(f'{scale}: alpha = {alpha:.3f}, 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]')